### Row Level Security(RLS):
_Row Level Security Ensures that differnt users see different rows of the same table based on permissions, roles or attributes_

In [0]:
%sql
SELECT * FROM databricksgovi.silver.stores_enr;

**USE CASE**
- _Here i want to apply the row level security for region column_
- _First i need to create rls_mapping table_
- _Second I need insert the records on it, based on the region. This is because of (_How the databricks know from which region I belongs to_)_
- _Next I will create a Masking_
- _Then I will create a Maksing Function & Apply it_

**Creating the rls mapping table**

In [0]:
%sql
CREATE TABLE databricksgovi.silver.rls_store_mapping
(
  user_id STRING,
  region STRING
)

**Inserting the records**

In [0]:
%sql
INSERT INTO databricksgovi.silver.rls_store_mapping VALUES
('postboxgovi@gmail.com', 'north'),
('rohan@gmail.com', 'south'),
('raghu@gmail.com', 'east'),
('shiv@gmail.com', 'west')

In [0]:
%sql
SELECT * FROM databricksgovi.silver.rls_store_mapping

In [0]:
%sql
SELECT * FROM databricksgovi.silver.rls_store_mapping
WHERE user_id = current_user()

**HERE I NEED DISPLAY THE STORES DATA AS PER REGION HE BELONGS TO**

_In our case i need to display on NORTH region stores_

In [0]:
%sql
-- Below SQL Statement RETURNS TRUE IF OR FALSE Based on the condition
SELECT EXISTS
(SELECT * FROM databricksgovi.silver.rls_store_mapping
WHERE user_id = current_user() AND
region = 'north')

**CREATING RLS FUNCTION**

In [0]:
%sql
CREATE OR REPLACE FUNCTION databricksgovi.silver.rls_store_func(p_region STRING)
RETURNS BOOLEAN
RETURN EXISTS
(SELECT * FROM databricksgovi.silver.rls_store_mapping
WHERE user_id = current_user() AND
region = lower(p_region))

**Creating Function on Region Column**

In [0]:
%sql
ALTER TABLE databricksgovi.silver.stores_enr
SET ROW FILTER databricksgovi.silver.rls_store_func ON (region)

In [0]:
%sql
SELECT * FROM databricksgovi.silver.stores_enr